### Create recipe jsonl and database files

In [1]:
import ast
import json
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

In [11]:
# Converts ../data/RAW_recipes.csv and ../data/RAW_interactions.csv into a single JSONL
# Each recipe record will include a "reviews" list with its interactions (if any).
DATA_DIR = Path("../data/recipes") / "raw_data"
RECIPE_CSV = DATA_DIR / "RAW_recipes.csv"
INTERACTION_CSV = DATA_DIR / "RAW_interactions.csv"
JOINED_JSONL = Path("../data/recipes/processed") /  "recipes_with_reviews.jsonl"

def read_recipes_csv(): 
    recipes =  pd.read_csv(RECIPE_CSV)
    recipes['nutrition'] = recipes['nutrition'].apply(lambda x: ast.literal_eval(x))
    recipes['steps'] = recipes['steps'].apply(lambda x: ast.literal_eval(x))
    recipes['ingredients'] = recipes['ingredients'].apply(lambda x: ast.literal_eval(x))
    recipes['tags'] = recipes['tags'].apply(lambda x: ast.literal_eval(x))
    reviews = pd.read_csv(INTERACTION_CSV)
    review_grpd = reviews.groupby('recipe_id').apply(lambda x: x.to_json(orient='records')).reset_index()
    review_grpd.rename(columns={review_grpd.columns[1]: 'reviews_json'}, inplace=True)
    reviews_col = []
    for index, row in tqdm(recipes.iterrows()):
        recipe_reviews = review_grpd[review_grpd['recipe_id'] == row['id']]
        reviews_col.append(json.loads(recipe_reviews.to_dict(orient='records')[0]['reviews_json']))
    recipes['reviews'] = reviews_col 
    return recipes

In [3]:
recipes = read_recipes_csv()

/var/folders/f0/wtxc51w57q13pv8mqkb715rr0000gp/T/ipykernel_68198/3423257914.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  review_grpd = reviews.groupby('recipe_id').apply(lambda x: x.to_json(orient='records')).reset_index()
231637it [00:54, 4278.64it/s]


In [4]:
recipes.head(5)

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,reviews
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"[60-minutes-or-less, time-to-make, course, mai...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"[make a choice and proceed with recipe, depend...",autumn is my favorite time of year to cook! th...,"[winter squash, mexican seasoning, mixed spice...",7,"[{'user_id': 4470, 'recipe_id': 137739, 'date'..."
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"[30-minutes-or-less, time-to-make, course, mai...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"[preheat oven to 425 degrees f, press dough in...",this recipe calls for the crust to be prebaked...,"[prepared pizza crust, sausage patty, eggs, mi...",6,"[{'user_id': 28603, 'recipe_id': 31490, 'date'..."
2,all in the kitchen chili,112140,130,196586,2005-02-25,"[time-to-make, course, preparation, main-dish,...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"[brown ground beef in large pot, add chopped o...",this modified version of 'mom's' chili was a h...,"[ground beef, yellow onions, diced tomatoes, t...",13,"[{'user_id': 255952, 'recipe_id': 112140, 'dat..."
3,alouette potatoes,59389,45,68585,2003-04-14,"[60-minutes-or-less, time-to-make, course, mai...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,[place potatoes in a large pot of lightly salt...,"this is a super easy, great tasting, make ahea...","[spreadable cheese with garlic and herbs, new ...",11,"[{'user_id': 296809, 'recipe_id': 59389, 'date..."
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"[weeknight, time-to-make, course, main-ingredi...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,"[mix all ingredients& boil for 2 1 / 2 hours ,...",my dh's amish mother raised him on this recipe...,"[tomato juice, apple cider vinegar, sugar, sal...",8,"[{'user_id': 1310146, 'recipe_id': 44061, 'dat..."


In [5]:
len(recipes)

231637

In [22]:
# Write to JSONL
jsonl_records = []
with open(JOINED_JSONL, 'w') as f:
    for _, row in tqdm(recipes.iterrows()):
        recipe = {}
        recipe['recipe_id'] = str(row['id'])
        recipe['name'] = row['name']
        recipe['description'] = str(row['description']).strip().replace('\n', '')
        recipe['cook_time_min'] = row['minutes']
        recipe['ingredients'] = row['ingredients']
        recipe['ingredient_count'] = row['n_ingredients']
        recipe['instructions'] = row['steps']
        recipe['tags'] = row['tags']
        recipe['n_steps'] = row['n_steps']
        recipe['reviews'] = row['reviews']
        recipe['review_count'] = len(row['reviews'])
        recipe['average_rating'] = np.mean([rev['rating'] for rev in recipe['reviews']])if recipe['review_count'] > 0 else None
        nutrition = row['nutrition']
        nutrition = {
            'calories': float(nutrition[0]),
            'total_fat_pdv': float(nutrition[1]),
            'sugar_pdv': float(nutrition[2]),
            'sodium_pdv': float(nutrition[3]),
            'protein_pdv': float(nutrition[4]),
            'saturated_fat_pdv': float(nutrition[5])
        }
        recipe['nutrition'] = nutrition
        recipe['display_text'] = """
        {name}\n
        Description: {description}\n
        Cook time: {cook_time} minutes\n
        Average Rating: {avg_rating}\n
        Number of Reviews: {n_reviews}\n
        Ingredients ({n_ingredients}): \t
        {ingredients}\n
        Nutrition per serving:
        Calories: {calories}
        Total Fat: {total_fat}% DV
        Saturated Fat: {saturated_fat}% DV
        Protein: {protein}% DV
        Sodium: {sodium}% DV
        Sugar: {sugar}% DV\n
        Instructions ({n_steps} steps): 
        {instructions}\n
        Tags: {tags} \n
        Reviews ({n_reviews}): 
        {reviews}\n
        """.format(name=recipe['name'],
                   description=recipe['description'],
                   cook_time=recipe['cook_time_min'],
                   avg_rating = recipe['average_rating'] if recipe['average_rating'] is not None else 'N/A',
                   n_ingredients=recipe['ingredient_count'],
                   ingredients='\n\t'.join(recipe['ingredients']),
                   n_steps=recipe['n_steps'],
                   tags=', '.join(recipe['tags']),
                   calories=nutrition['calories'],
                   total_fat=nutrition['total_fat_pdv'],
                   saturated_fat=nutrition['saturated_fat_pdv'],
                   protein=nutrition['protein_pdv'],
                   sodium=nutrition['sodium_pdv'],
                   sugar=nutrition['sugar_pdv'],
                   instructions='\n\t'.join([f"{i+1}. {item}" for i, item in enumerate(recipe['instructions'])]),
                   n_reviews=recipe['review_count'],
                   reviews='\n\t'.join([f"Rating: {rev['rating']}, Review: {rev['review']}" for rev in recipe['reviews']])
                  )
        jsonl_records.append(recipe)
        #print(recipe['display_text'])
        f.write(json.dumps(recipe))
        f.write('\n')

231637it [00:31, 7423.40it/s] 


### To CSV files for DB

In [23]:
formatted_recipes = pd.DataFrame(jsonl_records)
len(formatted_recipes)


231637

In [22]:
TABLES_DATA_DIR = Path("../data/recipes/processed/tables")


In [9]:
ingredients = set()
for _, row in formatted_recipes.iterrows():
    for ingredient in row['ingredients']:
        ingredients.add(ingredient.lower().strip())
ingredients_ls = []
for i, row in enumerate(ingredients):
    ingredient_id = i+1
    ingredients_ls.append({'ingredient_id': ingredient_id, 'ingredient_name': row})
ingredients_df = pd.DataFrame(ingredients_ls)
ingredients_df.head(5)

,ingredient_id,ingredient_name
0,1,mango
1,2,vanilla pod
2,3,split peas
3,4,chinese jujube
4,5,chile jelly


In [10]:
ingredients_df.to_csv(TABLES_DATA_DIR / "ingredients.csv", index=False)

In [11]:
recipes_df = []

for _, row in tqdm(formatted_recipes.iterrows()):
    recipe_record = {
        'recipe_id': row['recipe_id'],
        'name': row['name'],
        'description': row['description'],
        'cook_time_min': row['cook_time_min'],
        'ingredient_count': row['ingredient_count'],
        'n_steps': row['n_steps'],
        'review_count': row['review_count'],
        'average_rating': row['average_rating'],
        'instructions': row['instructions'],
        'display_text': row['display_text']
    }
    recipes_df.append(recipe_record)
recipes_df = pd.DataFrame(recipes_df)
recipes_df.head(5)

231637it [00:06, 33621.36it/s]


,recipe_id,name,description,cook_time_min,ingredient_count,n_steps,review_count,average_rating,instructions,display_text
0,137739,arriba baked winter squash mexican style,autumn is my favorite time of year to cook! th...,55,7,11,3,5.0,"[make a choice and proceed with recipe, depend...",\n arriba baked winter squash mexican...
1,31490,a bit different breakfast pizza,this recipe calls for the crust to be prebaked...,30,6,9,4,3.5,"[preheat oven to 425 degrees f, press dough in...",\n a bit different breakfast pizza\n\n...
2,112140,all in the kitchen chili,this modified version of 'mom's' chili was a h...,130,13,6,1,4.0,"[brown ground beef in large pot, add chopped o...",\n all in the kitchen chili\n\n ...
3,59389,alouette potatoes,"this is a super easy, great tasting, make ahea...",45,11,11,2,4.5,[place potatoes in a large pot of lightly salt...,\n alouette potatoes\n\n Descri...
4,44061,amish tomato ketchup for canning,my dh's amish mother raised him on this recipe...,190,8,5,1,5.0,"[mix all ingredients& boil for 2 1 / 2 hours ,...",\n amish tomato ketchup for canning\n...


In [12]:
recipes_df.to_csv(TABLES_DATA_DIR / "recipes.csv", index=False)

In [13]:
recipe_ingredients_ls = []
for _, row in tqdm(formatted_recipes.iterrows()):
    recipe_id = row['recipe_id']
    for ingredient in row['ingredients']:
        ingredient_name = ingredient.lower().strip()
        ingredient_id = ingredients_df[ingredients_df['ingredient_name'] == ingredient_name]['ingredient_id'].values[0]
        recipe_ingredients_ls.append({
            'recipe_id': recipe_id,
            'ingredient_id': ingredient_id
        })
recipe_ingredients_df = pd.DataFrame(recipe_ingredients_ls)
recipe_ingredients_df.head(5)

231637it [22:34, 171.04it/s]


,recipe_id,ingredient_id
0,137739,9594
1,137739,581
2,137739,1980
3,137739,12215
4,137739,11067


In [14]:
recipe_ingredients_df.to_csv(TABLES_DATA_DIR / "recipe_ingredients.csv", index=False)

In [15]:
recipe_tags_ls = []
tag_ids_map = {}
next_tag_id = 1

for _, row in tqdm(formatted_recipes.iterrows()):
    for tag in row['tags']:
        if tag not in tag_ids_map:
            tag_ids_map[tag] = next_tag_id
            next_tag_id += 1
        recipe_tags_ls.append({
            'recipe_id': row['recipe_id'],
            'tag_id': tag_ids_map[tag]
        })

recipe_tags_df = pd.DataFrame(recipe_tags_ls)
recipe_tags_df.head(5)


231637it [00:14, 16280.55it/s]


,recipe_id,tag_id
0,137739,1
1,137739,2
2,137739,3
3,137739,4
4,137739,5


In [16]:

recipe_tags_df.to_csv(TABLES_DATA_DIR / "recipe_tags.csv", index=False)


In [17]:
tags_df = pd.DataFrame([{'tag_id': tag_id, 'tag_name': tag_name} for tag_name, tag_id in tag_ids_map.items()])
tags_df.head(5)

,tag_id,tag_name
0,1,60-minutes-or-less
1,2,time-to-make
2,3,course
3,4,main-ingredient
4,5,cuisine


In [18]:
tags_df.to_csv(TABLES_DATA_DIR / "tags.csv", index=False)

In [26]:
nutrition_labels = []
for _, row in tqdm(formatted_recipes.iterrows()):
    recipe_id = row['recipe_id']
    nutrition = row['nutrition']
    nutrition_row = {
            'recipe_id': recipe_id,
            'calories': nutrition['calories'],
            'total_fat_pdv':  nutrition['total_fat_pdv'],
            'sugar_pdv': nutrition['sugar_pdv'],
            'sodium_pdv':  nutrition['sodium_pdv'],
            'protein_pdv': nutrition['protein_pdv'],
            'saturated_fat_pdv': nutrition['saturated_fat_pdv']
        }
    nutrition_labels.append(nutrition_row)
nutrition_df = pd.DataFrame(nutrition_labels)
nutrition_df.head(5)

231637it [00:05, 46298.37it/s]


,recipe_id,calories,total_fat_pdv,sugar_pdv,sodium_pdv,protein_pdv,saturated_fat_pdv
0,137739,51.5,0.0,13.0,0.0,2.0,0.0
1,31490,173.4,18.0,0.0,17.0,22.0,35.0
2,112140,269.8,22.0,32.0,48.0,39.0,27.0
3,59389,368.1,17.0,10.0,2.0,14.0,8.0
4,44061,352.9,1.0,337.0,23.0,3.0,0.0


In [27]:
nutrition_df.to_csv(TABLES_DATA_DIR / "nutrition.csv", index=False)


## Recipe Tag Categories

In [ ]:
tags_df = pd.read_csv(TABLES_DATA_DIR/"tags.csv")
tags_categories = json.load(open(Path("../data/recipes/processed/recipe_categories.json"), 'r'))

#sanity check

for category in tags_categories:
    print(f"Category: {category['category']}")
    for tag in category['tags']:
        tag_name = tag['tag_name']
        if tag_name not in tags_df['tag_name'].values:
            print(f"  Missing tag: {tag_name}")
            break
        else:
            if tag['tag_id'] != tags_df[tags_df['tag_name'] == tag_name]['tag_id'].values[0]:
                print(f"  Mismatched tag ID for {tag_name}: expected {tag['tag_id']}")
                break
            else:
                continue
print("  All tags present and correct.")
            